# GraphRAG over the *12 Angry Men* Screenplay

**Author:** Giannis Mertziotis · **Co-author:** Spyridon Tzokas

Part 3 of a two-part AI lab project (NTUA, ECE). This notebook implements **retrieval over a pre-built knowledge graph** extracted from the screenplay of *12 Angry Men*, following the GraphRAG approach.

**Inputs (given):**

- `graph.json` — a cleaned knowledge graph with entities as nodes and relationships as edges.
- `community_summaries.json` — optional high-level GraphRAG community summaries.
- `questions.csv` — indicative test questions.

**Approach.** For each question we locate relevant nodes/edges, expand around them with **k-hop traversal** over a NetworkX graph, serialize the retrieved subgraph into text evidence, optionally add community summaries, and pass everything to **Qwen3-4B-Instruct** to produce an answer. Results are saved to CSV (one run with community summaries, one without).


## How the graph was created

The graph was created in a separate notebook. The high-level process was:

1. **Load the screenplay** and split it into screenplay-aware chunks.
2. **Use an LLM-based knowledge graph extractor** to identify entities and relationships.
3. **Store the extracted graph** in a LlamaIndex `PropertyGraphIndex` / `SimplePropertyGraphStore`.
4. **Convert the LlamaIndex graph to NetworkX** so that the graph can be inspected, cleaned, visualized, and exported.
5. **Clean the graph** by removing isolated nodes and obvious noisy autogenerated nodes.
6. **Save the final graph** as `graph.json`.
7. Optionally, **detect graph communities** and summarize each community. These summaries are saved in `community_summaries.json`.

This assignment starts **after** the graph has already been created. Therefore, we focus on retrieval: given a question, how do we find the most relevant nodes, edges, and community summaries?

## Exercise pipeline

For each question, we implement this retrieval-and-answering pipeline:

```text
Question
   ↓
Find relevant graph nodes and edges
   ↓
Expand around the matched nodes with k-hop graph traversal
   ↓
Serialize the retrieved subgraph into text evidence
   ↓
Optionally retrieve relevant community summaries
   ↓
Give the retrieved evidence to Qwen3-4B-Instruct
   ↓
Save the generated answer in a CSV file
```

We do **not** evaluate the answer automatically against the gold answer in this notebook. The CSV contains the correct answer only for human inspection.

## 1. Install dependencies

In [ ]:
!pip -q install networkx pandas scikit-learn tqdm transformers accelerate sentencepiece sentence-transformers scikit-learn networkx pandas tqdm

## 2. Imports and configuration

Place the files in the same directory as this notebook, or update the paths below.

In [ ]:
from pathlib import Path
import json
import re

import networkx as nx
import pandas as pd
from tqdm.auto import tqdm

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
# Input files
GRAPH_JSON_PATH = Path("graph.json")
COMMUNITY_SUMMARIES_PATH = Path("community_summaries.json")
QUESTIONS_CSV_PATH = Path("questions.csv")

# Output file
OUTPUT_CSV_PATH = Path("generated_answers.csv")

## 3. Loading the Graph Data into NetworkX

In this cell, we take our raw data—stored in a JSON file—and convert it into a structured **graph** object that we can analyze and manipulate using Python.

Here is a high-level breakdown of the libraries and logic used in the code:

**Key Libraries Used:**
*   **`json`**: A built-in Python library used to read the file and convert the JSON text into Python dictionaries and lists.
*   **`networkx` (imported as `nx`)**: A powerful and widely used Python library for studying complex networks and graphs.
    *   Specifically, we use `nx.DiGraph()`, which initializes a **Directed Graph**. This means the connections between our nodes have a specific direction (like a one-way street from a "source" to a "target").

**Step-by-Step Code Explanation:**
1.  **Reading the File:** The function opens the specified JSON file and loads all its data into a dictionary called `data`.
2.  **Initializing the Graph:** We create an empty directed graph (`G = nx.DiGraph()`) ready to be populated.
3.  **Adding Nodes (The Entities):**
    *   The code loops through the `"nodes"` list in our JSON.
    *   It extracts the fundamental attributes like `id`, `name`, and `entity_type`.
    *   *Smart Fallbacks:* Notice the use of `.get() ` and `or`. This makes our code robust; if a node is missing a specific "name", the code safely falls back to using the "id" instead of crashing.
    *   Any extra metadata is bundled together and added as properties attached to the node using `**props`.
4.  **Adding Edges (The Relationships):**
    *   Next, it loops through the `"edges"` list using `enumerate()` (which gives us a helpful index `i` to use as a fallback ID if one isn't provided).
    *   It identifies the `source` node and the `target` node for every connection.
    *   Just like with nodes, it extracts the `relation` type and `description`, applying safe fallbacks if the data is slightly messy or missing.
    *   Finally, it adds the edge to the graph using `G.add_edge()`.
5.  **Verification:** Once the graph is built and returned, we print out the total number of nodes and edges to confirm everything loaded successfully.

In [ ]:
def load_graph_json(path: Path) -> nx.DiGraph:
    # Load graph_clean.json into a NetworkX DiGraph.
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)

    G = nx.DiGraph()

    for node in data["nodes"]:
        node_id = str(node["id"])
        props = dict(node.get("properties", {}))
        props.pop("name", None)
        props.pop("entity_type", None)
        G.add_node(
            node_id,
            name=str(node.get("name") or node_id),
            entity_type=str(node.get("entity_type") or "entity"),
            **props,
        )

    for i, edge in enumerate(data["edges"]):
        source = str(edge["source"])
        target = str(edge["target"])
        props = dict(edge.get("properties", {}))
        relation = str(edge.get("relation") or props.get("relation") or "RELATED_TO")
        description = str(edge.get("description") or props.get("description") or props.get("relationship_description") or "")
        props.pop("relation", None)
        props.pop("description", None)
        props.pop("relationship_description", None)
        G.add_edge(
            source,
            target,
            relation=relation,
            description=description,
            edge_id=str(props.get("rel_id") or i),
            **props,
        )

    return G

G = load_graph_json(GRAPH_JSON_PATH)
print("Graph loaded")
print("Nodes:", G.number_of_nodes())
print("Edges:", G.number_of_edges())

Graph loaded
Nodes: 77
Edges: 220


## 4. Inspect the graph

In this cell, we analyze the graph we just built to find the most "popular" or highly connected entities. We do this by calculating each node's **degree** (the total number of incoming and outgoing connections) and presenting the top results in a clean table.

**Key Libraries and Functions Used:**
*   **`G.degree()` (from NetworkX):** This calculates how many edges (relationships) are attached to each node. It returns pairs of data looking like `(node_id, degree)`.
*   **`sorted()` (Built-in Python):** We use this to arrange our nodes.
    *   The `key=lambda x: x[1]` part is a quick, anonymous function telling Python, "Sort these pairs based on the second item (the degree), not the first item (the ID)."
    *   `reverse=True` ensures we sort from the highest degree down to the lowest.
*   **`pandas` (imported as `pd`):** A powerful data analysis library. We use `pd.DataFrame()` to turn our raw list of node information into a beautifully formatted, easy-to-read data table.

**Step-by-Step Code Explanation:**
1.  **Setting the Limit:** The function `graph_overview` takes our graph `G` and an optional `top_n` parameter (defaulting to 15), meaning we only want to look at the top 15 most connected nodes by default.
2.  **Sorting and Selecting:** It calculates the degree for every single node, sorts them from highest to lowest, and uses list slicing (`[:top_n]`) to grab just the top group.
3.  **Gathering Node Data:** For each of those top nodes, it looks back into the graph's memory (`G.nodes[node_id]`) to retrieve its readable `"name"` and `"entity_type"`. It uses the `.get()` method as a safe fallback just in case a node is missing that specific information.
4.  **Building the Rows:** It compiles the ID, name, type, and degree into a dictionary, which represents a single row of data, and adds it to our `rows` list.
5.  **Final Output:** It converts that list of rows into a Pandas DataFrame, displaying our most central graph entities in a neat spreadsheet-like format

In [ ]:
def graph_overview(G: nx.DiGraph, top_n: int = 15):
    rows = []
    for node_id, degree in sorted(G.degree(), key=lambda x: x[1], reverse=True)[:top_n]:
        rows.append({
            "node_id": node_id,
            "name": G.nodes[node_id].get("name", node_id),
            "entity_type": G.nodes[node_id].get("entity_type", ""),
            "degree": degree,
        })
    return pd.DataFrame(rows)

graph_overview(G)

,node_id,name,entity_type,degree
0,JUROR #8,JUROR #8,entity,39
1,FOREMAN,FOREMAN,entity,37
2,JUROR #4,JUROR #4,entity,27
3,JUROR #3,JUROR #3,entity,26
4,JUROR #10,JUROR #10,CHARACTER,21
5,JUROR #7,JUROR #7,entity,18
6,JUROR #11,JUROR #11,entity,16
7,JUROR #9,JUROR #9,CHARACTER,15
8,#3,#3,entity,15
9,#8,#8,entity,15


### Count edge appearance

In [ ]:
relation_counts = pd.Series([data.get("relation", "") for _, _, data in G.edges(data=True)]).value_counts()
relation_counts.head(20).to_frame("count")

,count
QUESTIONS,17
SUPPORTS,10
ARGUES_WITH,10
ADDRESSES,9
CHALLENGES,7
CALMS,6
CONTRADICTS,6
RESPONDS_TO,5
DISCUSSES,4
AGREES_WITH,4


### Graph Inspection and Correction

Before diving into deep analysis, it is crucial to verify the integrity of our data. We highly encourage you to inspect the raw JSON file, explore the node/edge properties, and even visualize the graph to build an intuition for its structure. Since this is a relatively small network, you can explore it quite easily visually

**Your Task:**
1. **Inspect:** Programmatically or visually check for inconsistencies, missing properties, or isolated nodes.
2. **Identify:** Look for potential errors, such as duplicate nodes representing the exact same entity under slightly different IDs or names.
3. **Correct:** Use NetworkX functions to clean the graph (e.g., merge duplicates, remove invalid edges, or fill in missing data).

*Write your inspection and cleaning logic in the code cell below.*

In [ ]:
# =========================================================
# Graph Inspection and Correction
# =========================================================
# The graph 'G' is already loaded in memory.

# --- 1. INSPECTION ---
# Tip: You can visualize the graph using nx.draw(G, with_labels=True)
# or iterate through nodes/edges to find missing data.

print("Original Graph Size --> Nodes:", G.number_of_nodes(), "| Edges:", G.number_of_edges())



Original Graph Size --> Nodes: 77 | Edges: 220


In [ ]:
# --- 2. IDENTIFY DUPLICATES / INCONSISTENCIES ---
# Write your code here to find issues.
# Idea: Can you find nodes with the exact same name but different IDs?

import re
from collections import defaultdict

# 1. Find exact name duplicates (different IDs, same name)
name_to_ids = defaultdict(list)
for node_id, data in G.nodes(data=True):
    name = data.get("name", node_id).strip().lower()
    name_to_ids[name].append(node_id)

exact_dupes = {name: ids for name, ids in name_to_ids.items() if len(ids) > 1}
print("=== EXACT NAME DUPLICATES ===")
for name, ids in exact_dupes.items():
    print(f"  '{name}' -> IDs: {ids}")

# 2. Find shorthand duplicates: "#3" vs "JUROR #3"
print("\n=== SHORTHAND DUPLICATES (#X vs JUROR #X) ===")
shorthand_pairs = []
for node_id, data in G.nodes(data=True):
    name = data.get("name", node_id).strip()
    if re.match(r'^#\d+$', name):
        full_name = "juror " + name.lower()
        for other_id, other_data in G.nodes(data=True):
            other_name = other_data.get("name", "").strip().lower()
            if other_name == full_name and other_id != node_id:
                shorthand_pairs.append((node_id, name, other_id, other_data.get("name", "")))
                print(f"  '{name}' ({node_id}, deg={G.degree(node_id)}) <-> '{other_data.get('name','')}' ({other_id}, deg={G.degree(other_id)})")

# 3. Inconsistent entity_type
print("\n=== INCONSISTENT ENTITY TYPES ===")
type_counts = defaultdict(int)
for _, data in G.nodes(data=True):
    type_counts[data.get("entity_type", "MISSING")] += 1
for t, c in type_counts.items():
    print(f"  {t}: {c} nodes")

# 4. Isolated nodes
isolated = list(nx.isolates(G))
print(f"\n=== ISOLATED NODES: {len(isolated)} ===")
for nid in isolated:
    print(f"  {nid}: {G.nodes[nid].get('name', nid)}")




=== EXACT NAME DUPLICATES ===

=== SHORTHAND DUPLICATES (#X vs JUROR #X) ===
  '#3' (#3, deg=15) <-> 'JUROR #3' (JUROR #3, deg=26)
  '#9' (#9, deg=4) <-> 'JUROR #9' (JUROR #9, deg=15)
  '#10' (#10, deg=11) <-> 'JUROR #10' (JUROR #10, deg=21)
  '#8' (#8, deg=15) <-> 'JUROR #8' (JUROR #8, deg=39)
  '#4' (#4, deg=6) <-> 'JUROR #4' (JUROR #4, deg=27)
  '#11' (#11, deg=4) <-> 'JUROR #11' (JUROR #11, deg=16)
  '#12' (#12, deg=6) <-> 'JUROR #12' (JUROR #12, deg=8)
  '#5' (#5, deg=6) <-> 'JUROR #5' (JUROR #5, deg=14)
  '#6' (#6, deg=4) <-> 'JUROR #6' (JUROR #6, deg=9)
  '#2' (#2, deg=5) <-> 'JUROR #2' (JUROR #2, deg=10)
  '#7' (#7, deg=2) <-> 'JUROR #7' (JUROR #7, deg=18)

=== INCONSISTENT ENTITY TYPES ===
  entity: 73 nodes
  CHARACTER: 4 nodes

=== ISOLATED NODES: 0 ===


In [ ]:
# --- 3. CORRECTION ---
# Write your code here if you found any issues.
# Helpful NetworkX functions:
# - G.remove_node(node_id)
# - G.remove_edge(u, v)
# - nx.contracted_nodes(G, node_to_keep, node_to_merge) # Great for merging duplicates!

# <YOUR CODE HERE>
# 1. Merge shorthand duplicates: "#3" -> "JUROR #3" etc.
for node_id, data in list(G.nodes(data=True)):
    name = data.get("name", node_id).strip()
    if re.match(r'^#\d+$', name):
        full_name = "juror " + name.lower()
        for other_id, other_data in list(G.nodes(data=True)):
            other_name = other_data.get("name", "").strip().lower()
            if other_name == full_name and other_id != node_id:
                if node_id in G and other_id in G:
                    print(f"Merging '{name}' ({node_id}) into '{other_data.get('name','')}' ({other_id})")
                    G = nx.contracted_nodes(G, other_id, node_id, self_loops=False)

# 2. Merge exact name duplicates (if any remain)
name_to_ids = defaultdict(list)
for node_id, data in G.nodes(data=True):
    name = data.get("name", node_id).strip().lower()
    name_to_ids[name].append(node_id)

for name, ids in name_to_ids.items():
    if len(ids) > 1:
        ids_sorted = sorted(ids, key=lambda x: G.degree(x), reverse=True)
        keep = ids_sorted[0]
        for merge in ids_sorted[1:]:
            if merge in G and keep in G:
                print(f"Merging duplicate '{merge}' into '{keep}' (name: {name})")
                G = nx.contracted_nodes(G, keep, merge, self_loops=False)

# 3. Fix inconsistent entity_type -> all to "CHARACTER" for jurors
for node_id, data in G.nodes(data=True):
    name = data.get("name", node_id).strip().upper()
    if "JUROR" in name or "FOREMAN" in name:
        G.nodes[node_id]["entity_type"] = "CHARACTER"

# 4. Remove isolated nodes
isolated = list(nx.isolates(G))
if isolated:
    print(f"Removing {len(isolated)} isolated nodes")
    G.remove_nodes_from(isolated)



# --- VERIFICATION ---
print("Cleaned Graph Size  --> Nodes:", G.number_of_nodes(), "| Edges:", G.number_of_edges())
# --- ENTITY TYPE VERIFICATION ---
from collections import Counter
types = Counter(data.get("entity_type", "MISSING") for _, data in G.nodes(data=True))
for t, c in types.items():
    print(f"  {t}: {c}")

Merging '#3' (#3) into 'JUROR #3' (JUROR #3)
Merging '#9' (#9) into 'JUROR #9' (JUROR #9)
Merging '#10' (#10) into 'JUROR #10' (JUROR #10)
Merging '#8' (#8) into 'JUROR #8' (JUROR #8)
Merging '#4' (#4) into 'JUROR #4' (JUROR #4)
Merging '#11' (#11) into 'JUROR #11' (JUROR #11)
Merging '#12' (#12) into 'JUROR #12' (JUROR #12)
Merging '#5' (#5) into 'JUROR #5' (JUROR #5)
Merging '#6' (#6) into 'JUROR #6' (JUROR #6)
Merging '#2' (#2) into 'JUROR #2' (JUROR #2)
Merging '#7' (#7) into 'JUROR #7' (JUROR #7)
Cleaned Graph Size  --> Nodes: 66 | Edges: 186
  CHARACTER: 13
  entity: 53


## 5. Convert graph content to searchable text

Firstly we load our embedding model

In [ ]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

### Translating the Graph into Plain Text

In this cell, we are converting our structured graph data into human-readable text "documents." This is a crucial preparation step if you want to feed this data into a Natural Language Processing (NLP) model, a search engine, or an LLM, as they understand plain text much better than raw JSON or graph objects.

**Key Libraries Used:**
*   **`re` (Regular Expressions):** A built-in Python library for pattern matching in strings. We use it here to clean up messy text by replacing multiple spaces or newlines with a single space.
*   **`networkx` (`nx`):** Used to access the data stored inside our nodes and edges.

**High-Level Workflow:**
1.  **Text Cleaning (`clean_text`):** A helper function that ensures our final strings are neat and properly formatted without weird spacing issues.
2.  **Node Translation (`node_to_text`):** This function grabs all the attributes of a specific node (like its name, type, and description) and stitches them together into a single, comprehensive descriptive sentence.
3.  **Edge Translation (`edge_to_text`):** Similarly, this function looks at a relationship between two nodes. It grabs the names of the source and target, along with the relationship description, and forms a sentence explaining how they connect.
4.  **Batch Processing:** Finally, the code loops through every single node and edge in the graph, applies the translation functions, and stores the resulting strings in `node_texts` and `edge_texts` lists.

By the end of this cell, instead of a mathematical network, we have two lists of clean, descriptive sentences ready for text analysis!

In [ ]:
import re
import networkx as nx


def clean_text(x) -> str:
    if x is None:
        return ""
    return re.sub(r"\s+", " ", str(x)).strip()


def node_to_text(G: nx.DiGraph, node_id: str) -> str:
    data = G.nodes[node_id]

    name = data.get("name", node_id)
    entity_type = data.get("entity_type", "")
    entity_description = data.get("entity_description", "")
    relationship_description = data.get("relationship_description", "")
    scene_heading = data.get("scene_heading", "")
    start_page = data.get("start_page", "")
    end_page = data.get("end_page", "")

    text = (
        f"Node name: {name}. "
        f"Entity type: {entity_type}. "
        f"Entity description: {entity_description}. "
        f"Related description: {relationship_description}. "
        f"Scene heading: {scene_heading}. "
        f"Pages: {start_page}-{end_page}."
    )

    return clean_text(text)


def edge_to_text(G: nx.DiGraph, source: str, target: str, data: dict) -> str:
    source_name = G.nodes[source].get("name", source)
    target_name = G.nodes[target].get("name", target)

    relation = data.get("relation", "")
    description = data.get("description", "") or data.get("relationship_description", "")

    start_page = data.get("start_page", "")
    end_page = data.get("end_page", "")

    text = (
        f"Source: {source_name}. "
        f"Target: {target_name}. "
        f"Relation: {relation}. "
        f"Description: {description}. "
        f"Pages: {start_page}-{end_page}."
    )

    return clean_text(text)


node_ids = list(G.nodes())
node_texts = [node_to_text(G, node_id) for node_id in node_ids]

edge_records = []
edge_texts = []

for source, target, data in G.edges(data=True):
    text = edge_to_text(G, source, target, data)
    edge_records.append((source, target, data, text))
    edge_texts.append(text)

print("Node documents:", len(node_texts))
print("Edge documents:", len(edge_texts))

if edge_texts:
    print("Example edge text:\n", edge_texts[0])

Node documents: 66
Edge documents: 186
Example edge text:
 Source: FOREMAN. Target: JURY. Relation: ANNOUNCES_RESULT. Description: The foreman states that the vote is nine to three in favor of acquittal.. Pages: 126-128.


## 6. Turning Text into Numbers (Vectorization & Embeddings)

To prepare our graph for tasks like search or question-answering, we must convert our lists of descriptive sentences into numerical representations. We are going to be using the same two techniques we already used in the previous assignment:


**1. TF-IDF (The Keyword-Based Approach)**
*   **Library:** `TfidfVectorizer` from `scikit-learn` (`sklearn`).
*   **How it works Reminder:** TF-IDF stands for Term Frequency-Inverse Document Frequency. It scores words based on how important they are. If a word appears often in one specific sentence but rarely across all others, it gets a high score (meaning it's a strong keyword for that specific node/edge).
*   **Smart filtering:** We use `stop_words="english"` to automatically ignore unhelpful words like "the", "is", or "and". We also use `ngram_range=(1, 2)` to look at both single words and pairs of words (e.g., capturing "New York" as a single concept).

**2. Semantic Embeddings (The Context-Based Approach)**
*   **Tool:** A pre-trained `embedding_model` (such as SentenceTransformers).
*   **How it works:** Unlike TF-IDF, which relies on exact word matches, embeddings understand *context and meaning*. It converts the text into a dense array of numbers where sentences with similar meanings end up mathematically close to each other, even if they use completely different words.
*   **Standardization:** We set `normalize_embeddings=True` to scale these vectors uniformly, which makes comparing them later mathematically much easier and faster.

**Step-by-Step Execution:**
The code creates these mathematical matrices for both our Node texts and Edge texts independently. Finally, we print the `.shape` of the resulting matrices. The output will look something like `(number_of_documents, number_of_features)`, confirming that our text has been successfully translated into a format the computer can compute

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np


# TF-IDF indexes
node_vectorizer = TfidfVectorizer(
    stop_words="english",
    ngram_range=(1, 2),
)

node_tfidf_matrix = node_vectorizer.fit_transform(node_texts)


edge_vectorizer = TfidfVectorizer(
    stop_words="english",
    ngram_range=(1, 2),
)

edge_tfidf_matrix = edge_vectorizer.fit_transform(edge_texts)


# Semantic embedding indexes
node_embedding_matrix = embedding_model.encode(
    node_texts,
    normalize_embeddings=True,
    show_progress_bar=True,
)

edge_embedding_matrix = embedding_model.encode(
    edge_texts,
    normalize_embeddings=True,
    show_progress_bar=True,
)


print("Node TF-IDF matrix:", node_tfidf_matrix.shape)
print("Edge TF-IDF matrix:", edge_tfidf_matrix.shape)
print("Node embedding matrix:", node_embedding_matrix.shape)
print("Edge embedding matrix:", edge_embedding_matrix.shape)

Batches:   0%|          | 0/3 [00:00<?, ?it/s]

Batches:   0%|          | 0/6 [00:00<?, ?it/s]

Node TF-IDF matrix: (66, 818)
Edge TF-IDF matrix: (186, 2336)
Node embedding matrix: (66, 384)
Edge embedding matrix: (186, 384)


### Processing Community Summaries (The GraphRAG way)

In this cell, we handle an optional but powerful feature: **Community Summaries**. In large networks, nodes naturally form dense clusters or "communities." In the classic GraphRAG Retrieval system the retrieval is not happening at all from the graph but from the community summaries (again generated by a LLM). We have created what the actual GraphRAG code would produce for our specific example and have given you a community_summaries.json file. In our case that the script is very small and limited and there are many complex interractions, the summaries are not as useful as in other cases. You can use the community summaries as supportive in this case but note these summaries were written using the initial graph so in theory you would need to be generate them again if the graph has been altered. You can test whether the community summaries help or not in the answers anyway in this case.

**High-Level Logic:**
*   **The Toggle Switch (`community_summaries = False`):** By default, this is turned off. The code uses an `if/else` block to check if we actually have community data loaded into memory.
*   **If Turned On (The `if` block):** The code runs the exact same two-pronged vectorization process we saw in the previous cell. It uses TF-IDF to find community-level keywords and our Embedding Model to capture the overall semantic meaning of the cluster.
*   **If Turned Off (The `else` block):** The code safely sets our mathematical matrices to `None`. This acts as a graceful placeholder, preventing our notebook from crashing later on when it looks for these variables.

*Why do we care about this?* Vectorizing community summaries allows our future search tools to "zoom out." Instead of only looking for specific individual nodes, we can search for broad themes across entire neighborhoods of the graph

In [ ]:
community_ids = []
community_texts = []
# Change to true to use community summaries
community_summaries = True

# Load and enable community summaries
with open(COMMUNITY_SUMMARIES_PATH, "r", encoding="utf-8") as f:
    community_summaries = json.load(f)

if community_summaries:
    community_ids = list(community_summaries.keys())
    community_texts = [community_summaries[cid] for cid in community_ids]

    community_vectorizer = TfidfVectorizer(
        stop_words="english",
        ngram_range=(1, 2),
    )

    community_tfidf_matrix = community_vectorizer.fit_transform(community_texts)

    community_embedding_matrix = embedding_model.encode(
        community_texts,
        normalize_embeddings=True,
        show_progress_bar=True,
    )

    print("Community summaries:", len(community_texts))
    print("Community TF-IDF matrix:", community_tfidf_matrix.shape)
    print("Community embedding matrix:", community_embedding_matrix.shape)

else:
    community_vectorizer = None
    community_tfidf_matrix = None
    community_embedding_matrix = None

    print("No community summaries loaded.")

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Community summaries: 23
Community TF-IDF matrix: (23, 728)
Community embedding matrix: (23, 384)


## 7. Retrieval functions

### Assignment Task: The Full Graph Retrieval Pipeline

Now we are going to build the core engine of our application! When a user asks a question, we don't just want to find a single matching node. We want to find the exact neighborhood in the graph that holds the answer, extract it, and format it as evidence.

This process is a mini **Retrieval-Augmented Generation (RAG)** pipeline specifically designed for Knowledge Graphs.

Here is the step-by-step journey your code needs to execute:

**1. Query Expansion (Helping the Search Engine)**
Users often use different words than our graph does (e.g., asking about a "knife" when the graph says "switchblade"). You will start by checking the query against a dictionary of synonyms and expanding it to catch more matches.

**2. The Hybrid Search Engine**
Why choose between exact keywords and semantic meaning when we can have both like we did in Assignment 2? You will build a hybrid search function that calculates **both** the TF-IDF score and the Embedding score for the query against our data matrices. It will normalize these scores, combine them using predefined weights, and return the top $K$ results.

**3. Finding "Seed Nodes" (The Entry Points)**
Using your hybrid search, you will query both the *nodes* and the *edges* to find the best initial matches. The nodes involved in these top matches become our "Seed Nodes"—our starting points in the graph.

**4. Subgraph Expansion (Exploring the Neighborhood)**
A single node rarely tells the whole story. From your Seed Nodes, you will perform a **$k$-hop expansion**. This means grabbing the seed nodes *and* all the nodes connected to them within a certain distance (number of hops). This creates a localized "subgraph" containing the broader context.

**5. Re-ranking the Evidence (Filtering the Noise)**
When we expand our subgraph, we inevitably drag in some irrelevant connections (noise). To fix this, you will convert the subgraph's edges back into text sentences and run them through a *second* round of Hybrid Search against the user's query. This ensures only the most relevant evidence lines survive!

**6. Global Context (Community Summaries)**
Finally, you will optionally retrieve high-level Community Summaries to give the final system a "macro" view of the graph's themes.

**Your Goal:**
Follow the comments in the cell below to implement the missing logic for each step of this pipeline. By the end, the `retrieve_context()` function should take a plain English question and return a highly targeted block of textual evidence!

In [ ]:
import networkx as nx
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer

# =========================================================
# HYPERPARAMETERS (Do not change these while developing but then you can play with them to see how it affects retrieval)
# =========================================================
TOP_K_SEED_NODES = 5
TOP_K_SEED_EDGES = 8
TOP_K_EVIDENCE_LINES = 12
TOP_K_COMMUNITIES = 3
HOPS = 1

USE_COMMUNITY_SUMMARIES = True
KEYWORD_WEIGHT = 0.70
EMBEDDING_WEIGHT = 0.30

# =========================================================
# 1. QUERY EXPANSION & UTILITIES
# =========================================================
# Here you can customize you logic for the query expansion. In our case this is very important as the node have very specific names so the exact wording of the user's query is very important! You are free to use whatever you want for this, we are providing a basic case here as a template but you can do whatever you want!
QUERY_EXPANSIONS = {
    # Hard-coded examples
    "weather": "heat hot hottest rain rainy raining storm",
    "witness": "eyewitness testimony saw heard",
}

def expand_query(query: str) -> str:
    expanded_query = query
    query_lower = query.lower()
    for key, synonyms in QUERY_EXPANSIONS.items():
        if key in query_lower:
            expanded_query += " " + synonyms
    return expanded_query

    return expanded_query

def normalize_scores(scores):
    """Min-Max normalization. Brings array values between 0.0 and 1.0. (Provided)"""
    scores = np.asarray(scores, dtype=float)
    if len(scores) == 0: return scores
    min_score, max_score = scores.min(), scores.max()
    if max_score == min_score: return np.zeros_like(scores)
    return (scores - min_score) / (max_score - min_score)

def subgraph_to_evidence_lines(subgraph: nx.DiGraph):
    """Converts a NetworkX subgraph into readable text lines. (Provided)"""
    lines = []
    for source, target, data in subgraph.edges(data=True):
        lines.append(edge_to_text(subgraph, source, target, data))
    return lines

# =========================================================
# 2. CORE SEARCH ALGORITHM
# =========================================================
def hybrid_search(query, texts, ids, tfidf_vectorizer, tfidf_matrix, embedding_matrix, top_k, keyword_weight=KEYWORD_WEIGHT, embedding_weight=EMBEDDING_WEIGHT):
    results = []

    # 1. Expand the query
    expanded_query = expand_query(query)

    # 2. TF-IDF scores (εδω πέρα δεν πιάνουμε συνώνυμα)
    query_tfidf = tfidf_vectorizer.transform([expanded_query])
    keyword_scores = cosine_similarity(query_tfidf, tfidf_matrix).flatten()

    # 3. Embedding scores (εδω πιάνει γιατι τοποθετούνται κοντα λέξεις με παρόμοια σημασία )
    query_embedding = embedding_model.encode([expanded_query], normalize_embeddings=True)
    embedding_scores = cosine_similarity(query_embedding, embedding_matrix).flatten()

    # 4. Normalize
    keyword_scores_norm = normalize_scores(keyword_scores)
    embedding_scores_norm = normalize_scores(embedding_scores)

    # 5. Combine
    final_scores = keyword_weight * keyword_scores_norm + embedding_weight * embedding_scores_norm

    # 6. Sort and return top_k
    top_indices = np.argsort(final_scores)[::-1][:top_k]
    for idx in top_indices:
        results.append({
            "id": ids[idx],
            "score": float(final_scores[idx]),
            "keyword_score": float(keyword_scores[idx]),
            "embedding_score": float(embedding_scores[idx]),
            "text": texts[idx],
        })

    return results

# =========================================================
# 3. PIPELINE COMPONENTS
# =========================================================
#-----τρεχει hybrid search σε nodes & edges--------
def retrieve_seed_nodes(question, top_k_nodes=TOP_K_SEED_NODES, top_k_edges=TOP_K_SEED_EDGES):
    seed_nodes = set()
    debug = {}

    node_hits = hybrid_search(question, node_texts, node_ids,
                              node_vectorizer, node_tfidf_matrix, node_embedding_matrix, top_k_nodes)

    edge_ids_list = list(range(len(edge_texts)))
    edge_hits = hybrid_search(question, edge_texts, edge_ids_list,
                              edge_vectorizer, edge_tfidf_matrix, edge_embedding_matrix, top_k_edges)

    for hit in node_hits:
        seed_nodes.add(hit["id"])

    for hit in edge_hits:
        source, target, data, text = edge_records[hit["id"]]
        seed_nodes.add(source)
        seed_nodes.add(target)

    debug["expanded_query"] = expand_query(question)
    debug["node_hits"] = node_hits
    debug["edge_hits"] = edge_hits

    return list(seed_nodes), debug

def expand_subgraph(G, seed_nodes, hops=HOPS):#κανει k hops για να εξερευνήσει k γείτονες
    G_undirected = G.to_undirected()
    expanded_nodes = set()
    for seed in seed_nodes:
        if seed in G_undirected:
            neighbors = nx.single_source_shortest_path_length(G_undirected, seed, cutoff=hops)
            expanded_nodes.update(neighbors.keys())
    return G.subgraph(expanded_nodes).copy()

def subgraph_to_evidence_lines(subgraph: nx.DiGraph):
    """
    Convert retrieved subgraph edges into textual evidence lines.
    """

    lines = []

    for source, target, data in subgraph.edges(data=True):
        lines.append(edge_to_text(subgraph, source, target, data))

    return lines

def rank_evidence_lines(question: str, evidence_lines, top_k: int = TOP_K_EVIDENCE_LINES):
    """
    Re-ranks sentences extracted from the subgraph to filter out noise.
    Outputs: ranked (list of strings)
    """
    ranked = []
    evidence_lines = [line for line in evidence_lines if clean_text(line)]

    if not evidence_lines:
        return []

    local_ids = list(range(len(evidence_lines)))

    local_vectorizer = TfidfVectorizer(
        stop_words="english",
        ngram_range=(1, 2),
    )

    local_tfidf_matrix = local_vectorizer.fit_transform(evidence_lines)

    local_embedding_matrix = embedding_model.encode(
        evidence_lines,
        normalize_embeddings=True,
    )

    hits = hybrid_search(question, evidence_lines, local_ids,
                         local_vectorizer, local_tfidf_matrix, local_embedding_matrix, top_k)

    seen = set()
    for hit in hits:
       for hit in hits:
        line = evidence_lines[hit["id"]]
        if line not in seen:
            seen.add(line)
            ranked.append(line)

    return ranked

def retrieve_context(question, use_communities=USE_COMMUNITY_SUMMARIES):
    seed_nodes, debug = retrieve_seed_nodes(question)
    subgraph = expand_subgraph(G, seed_nodes, hops=HOPS)
    evidence_lines = subgraph_to_evidence_lines(subgraph)
    ranked_evidence = rank_evidence_lines(question, evidence_lines, top_k=TOP_K_EVIDENCE_LINES)
    communities = retrieve_community_summaries(question) if use_communities else []

    context_parts = ["GRAPH EVIDENCE:"]
    if ranked_evidence:
        context_parts.extend(f"- {line}" for line in ranked_evidence)
    else:
        context_parts.append("No graph evidence retrieved.")

    if communities:
        context_parts.append("\nCOMMUNITY SUMMARIES:")
        for item in communities:
            context_parts.append(
                f"- Community {item['community_id']} "
                f"(score={item['score']:.3f}, "
                f"keyword={item['keyword_score']:.3f}, "
                f"embedding={item['embedding_score']:.3f}): "
                f"{item['summary']}"
            )

    return {
        "question": question,
        "seed_nodes": seed_nodes,
        "subgraph": subgraph,
        "evidence_lines": ranked_evidence,
        "communities": communities,
        "context": "\n".join(context_parts),
        "debug": debug,
    }
# =========================================================
# 4. THE MASTER PIPELINE
# =========================================================
def retrieve_community_summaries(question, top_k=TOP_K_COMMUNITIES):
    """
    Optional retrieval over precomputed community summaries.
    """
    if (
        not community_summaries
        or community_vectorizer is None
        or community_tfidf_matrix is None
        or community_embedding_matrix is None
    ):
        return []

    hits = hybrid_search(
        query=question,
        texts=community_texts,
        ids=community_ids,
        tfidf_vectorizer=community_vectorizer,
        tfidf_matrix=community_tfidf_matrix,
        embedding_matrix=community_embedding_matrix,
        top_k=top_k,
    )

    return [
        {
            "community_id": hit["id"],
            "score": hit["score"],
            "keyword_score": hit["keyword_score"],
            "embedding_score": hit["embedding_score"],
            "summary": community_summaries[hit["id"]],
        }
        for hit in hits
    ]
def retrieve_context(question: str, use_communities: bool = USE_COMMUNITY_SUMMARIES):
    """
    The master function that ties the RAG pipeline together.
    """
    # TODO: Uncomment and complete the pipeline once your functions are working!

    # seed_nodes, debug = retrieve_seed_nodes(question)

    # subgraph = expand_subgraph(G, seed_nodes, hops=HOPS)

    # evidence_lines = subgraph_to_evidence_lines(subgraph)

    # ranked_evidence = rank_evidence_lines(question, evidence_lines, top_k=TOP_K_EVIDENCE_LINES)

    # communities = retrieve_community_summaries(question) if use_communities else []

    # ... Format the outputs into context_parts string ...
    seed_nodes, debug = retrieve_seed_nodes(question)
    subgraph = expand_subgraph(G, seed_nodes, hops=HOPS)
    evidence_lines = subgraph_to_evidence_lines(subgraph)
    ranked_evidence = rank_evidence_lines(question, evidence_lines, top_k=TOP_K_EVIDENCE_LINES)
    communities = retrieve_community_summaries(question) if use_communities else []

    context_parts = ["GRAPH EVIDENCE:"]
    if ranked_evidence:
        context_parts.extend(f"- {line}" for line in ranked_evidence)
    else:
        context_parts.append("No graph evidence retrieved.")

    if communities:
        context_parts.append("\nCOMMUNITY SUMMARIES:")
        for item in communities:
            context_parts.append(
                f"- Community {item['community_id']} "
                f"(score={item['score']:.3f}, "
                f"keyword={item['keyword_score']:.3f}, "
                f"embedding={item['embedding_score']:.3f}): "
                f"{item['summary']}"
            )

    return {
        "question": question,
        "seed_nodes": seed_nodes,
        "subgraph": subgraph,
        "evidence_lines": ranked_evidence,
        "communities": communities,
        "context": "\n".join(context_parts),
        "debug": debug,
    }

## 8. Test retrieval on one question

This cell does not call the LLM. It only checks what evidence the retriever finds. Use it for debugging and understanding.

In [ ]:
test_question = "Trace the sequence of votes for Juror #9. Why does he change his initial vote?"

retrieved = retrieve_context(test_question)

print("Question:", test_question)

print("\nExpanded query:")
print(retrieved["debug"]["expanded_query"])

print("\nSeed nodes:")
for node_id in retrieved["seed_nodes"]:
    print("-", node_id, "|", G.nodes[node_id].get("name", node_id))

print("\nTop node hits:")
for hit in retrieved["debug"]["node_hits"]:
    print(
        f"- {hit['id']} | score={hit['score']:.3f} | "
        f"keyword={hit['keyword_score']:.3f} | "
        f"embedding={hit['embedding_score']:.3f}"
    )
    print(" ", hit["text"][:250])

print("\nTop edge hits:")
for hit in retrieved["debug"]["edge_hits"]:
    edge_idx = hit["id"]
    source, target, data, text = edge_records[edge_idx]

    print(
        f"- edge {edge_idx} | {source} -> {target} | "
        f"score={hit['score']:.3f} | "
        f"keyword={hit['keyword_score']:.3f} | "
        f"embedding={hit['embedding_score']:.3f}"
    )
    print(" ", text[:300])

print("\nRetrieved evidence:")
for line in retrieved["evidence_lines"]:
    print("-", line)

print("\nRetrieved communities:")
for item in retrieved["communities"]:
    print(
        f"- Community {item['community_id']} | "
        f"score={item['score']:.3f} | "
        f"keyword={item['keyword_score']:.3f} | "
        f"embedding={item['embedding_score']:.3f}"
    )

Question: Trace the sequence of votes for Juror #9. Why does he change his initial vote?

Expanded query:
Trace the sequence of votes for Juror #9. Why does he change his initial vote?

Seed nodes:
- NOT GUILTY | NOT GUILTY
- JUROR #9 | JUROR #9
- JUROR #7 | JUROR #7
- JUROR #5 | JUROR #5
- FOREMAN | FOREMAN
- JUROR #4 | JUROR #4
- JUROR #2 | JUROR #2
- GUILTY | GUILTY
- JUROR #3 | JUROR #3
- JUROR #6 | JUROR #6
- ARGUMENT | ARGUMENT
- JURY VOTE | JURY VOTE

Top node hits:
- JUROR #9 | score=0.930 | keyword=0.186 | embedding=0.525
  Node name: JUROR #9. Entity type: CHARACTER. Entity description: A juror who observes that the vote is eleven to one.. Related description: . Scene heading: EXT. LONG SHOT - N. Y. - COURT OF GENERAL SESSIONS - DAY. Pages: 141-144.
- GUILTY | score=0.881 | keyword=0.206 | embedding=0.349
  Node name: GUILTY. Entity type: entity. Entity description: . Related description: Juror #2 votes not guilty after hesitating.. Scene heading: EXT. LONG SHOT - N. Y. - COU

## 9. Load Qwen3-4B-Instruct

In Colab, use a GPU runtime. `bfloat16` is usually suitable on modern GPUs. If you face hardware issues, try replacing `torch.bfloat16` with `torch.float16`.

In [ ]:
import torch
from transformers import pipeline

model_id = "Qwen/Qwen3-4B-Instruct-2507"

llm_pipeline = pipeline(
    "text-generation",
    model=model_id,
    model_kwargs={"torch_dtype": torch.bfloat16},
    device_map="auto",
)

config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!
`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/238 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

## 10. Answer generation

We have tested the system with this simple prompt but you can change it and experiment with it as you wish.

In [ ]:
def build_prompt(question: str, context: str) -> str:
    messages = [
        {
            "role": "system",
            "content": (
                "You answer questions about the screenplay of 12 Angry Men. "
                "Use only the provided graph evidence and community summaries. "
                "Do not invent facts. If the evidence is insufficient, say that the graph does not contain enough evidence. "
                "Give a concise answer and mention the most relevant evidence when possible."
            ),
        },
        {
            "role": "user",
            "content": f"Question:\n{question}\n\nRetrieved context:\n{context}\n\nAnswer:",
        },
    ]

    tokenizer = llm_pipeline.tokenizer
    if hasattr(tokenizer, "apply_chat_template"):
        return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

    return f"System: {messages[0]['content']}\n\nUser: {messages[1]['content']}\n\nAssistant:"


def generate_answer(question: str, context: str, max_new_tokens: int = 300) -> str:
    prompt = build_prompt(question, context)
    output = llm_pipeline(prompt, max_new_tokens=max_new_tokens, do_sample=False, return_full_text=False)
    return output[0]["generated_text"].strip()

## 11. Test the full pipeline on one question

In [ ]:
question="Which evidence of the case are brought by the guard for the Jurors to see again?"
retrieved = retrieve_context(question)
answer = generate_answer(question, retrieved["context"])

print("Question:")
print(question)
print("\nAnswer:")
print(answer)
print("\nContext used:")
print(retrieved["context"][:3000])

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Question:
Which evidence of the case are brought by the guard for the Jurors to see again?

Answer:
The guard does not bring any evidence of the case for the jurors to see again. The graph evidence shows that the guard counts the jurors as they enter and interacts with them in a procedural capacity (e.g., escorting Juror #3, responding to the foreman), but there is no mention of the guard presenting or informing the jurors about physical evidence or testimony. The actual evidence (such as the knife, glasses, or eyewitness testimony) is brought by jurors (e.g., Juror #4, Juror #6) or discussed by jurors, not by the guard. Therefore, the guard does not present any case evidence to the jurors.

Context used:
GRAPH EVIDENCE:
- Source: GUARD. Target: JURY. Relation: INFORMS. Description: The guard tells the jurors that everyone is present and that he is just outside if they need anything.. Pages: 6-9.
- Source: GUARD. Target: THE JURY. Relation: COUNTS. Description: The guard counts the jur

## 12. Batch testing from `questions.csv`

The CSV should contain:

```text
Question,Category,Correct Answer,Page References,Explanation
```

This notebook does not score the model answer automatically. It only saves the generated answer and the retrieved evidence.

In [ ]:
if not QUESTIONS_CSV_PATH.exists():
    example_df = pd.DataFrame([
        {
            "Question": "How does the weather change from the beginning of the trial to the end?",
            "Category": "Reasoning/Graph RAG",
            "Correct Answer": "The weather transitions from extreme heat at the beginning to a heavy rainstorm by the end.",
            "Page References": "6, 148, 149",
            "Explanation": "In page 6 Juror #7 states that this is the hottest day of the year. In pages 148, 149 it is heavily raining.",
        }
    ])
    example_df.to_csv(QUESTIONS_CSV_PATH, index=False)
    print(f"Created example file: {QUESTIONS_CSV_PATH}")
else:
    print(f"Found file: {QUESTIONS_CSV_PATH}")

Found file: questions.csv


In [ ]:
questions_df = pd.read_csv(QUESTIONS_CSV_PATH)
questions_df.head()

,Question,Category,Correct Answer,Page References,Explanation
0,How does the weather change from the beginning...,Reasoning/Graph RAG,The weather transitions from extreme heat at t...,"6, 148, 149",In page 6 Juror #7 states that this is the hot...
1,How do the jurors reassess the credibility of ...,Reasoning/Graph RAG,"At first, the boy's inability to remember the ...","25, 109, 110, 111, 112, 113",The reasoning depends on connecting the initia...
2,What detail about the woman's eyesight makes i...,Reasoning/Graph RAG,Juror 9 notices she has marks from glasses on ...,"118, 119, 120",The system must link the marks on her nose to ...
3,Which piece of evidence first seems very speci...,Reasoning/Graph RAG,The switchblade knife,"39, 40, 42, 43, 44",Pages 39 and 40 present the knife as rare and ...
4,How does Juror #8 use the physical evidence of...,Reasoning/Graph RAG,Juror #8 argues that the murder weapon is not ...,"41, 42, 43",Requires linking Juror #4's assertion about th...


In [ ]:
def answer_question_row(row):
    question = row["Question"]
    retrieved = retrieve_context(question)
    model_answer = generate_answer(question, retrieved["context"])

    return {
        "Model Answer": model_answer,
        "Retrieved Evidence": "\n".join(retrieved["evidence_lines"]),
        "Seed Nodes": ", ".join([G.nodes[n].get("name", n) for n in retrieved["seed_nodes"]]),
        "Retrieved Communities": ", ".join([str(item["community_id"]) for item in retrieved["communities"]]),
    }

results = []
for _, row in tqdm(questions_df.iterrows(), total=len(questions_df)):
    generated = answer_question_row(row)
    results.append({**row.to_dict(), **generated})

results_df = pd.DataFrame(results)
results_df.to_csv(OUTPUT_CSV_PATH, index=False)

print(f"Saved results to: {OUTPUT_CSV_PATH}")
results_df.head()

  0%|          | 0/11 [00:00<?, ?it/s]

Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both

## 13. Suggested experiments

After the baseline works, try changing one thing at a time:

1. Set `HOPS = 2`. Does recall improve, or does the context become noisier?
2. Set `USE_COMMUNITY_SUMMARIES = False`. Are graph edges alone enough?
3. Increase or decrease `TOP_K_EVIDENCE_LINES`.
4. Inspect failed questions and check whether the necessary evidence exists in the graph.
5. Add or edit a small number of graph edges manually and rerun retrieval.
6. Compare local graph evidence against community-summary evidence.

The goal is to understand how graph structure changes retrieval behavior.

In [ ]:
# === Experiment 1: HOPS = 2 ===
HOPS = 2
results_hops2 = []
for _, row in tqdm(questions_df.iterrows(), total=len(questions_df)):
    generated = answer_question_row(row)
    results_hops2.append({**row.to_dict(), **generated})

results_hops2_df = pd.DataFrame(results_hops2)
results_hops2_df.to_csv("generated_answers_hops2.csv", index=False)
HOPS = 1  # Reset

print("HOPS=2 done!")